# sEMG Prosthetic Gesture Classification
## Notebook 12: Explainable AI (XAI) & Model Interpretability (Google Colab Edition)

**Author:** Principal ML Scientist, Explainable AI Researcher, Biomedical Signal Processing Expert & Senior Software Architect  
**Project:** Machine Learning-Based sEMG Prosthetic Gesture Classification  

---

### Research Objective & Scope
Interpret the predictions of the final hyperparameter-optimized gesture classification model (**CatBoost**) using Explainable AI (XAI) techniques:
- **Global Explanations**: TreeSHAP summary, mean absolute SHAP values, feature importance rankings.
- **Local Explanations**: Waterfall plots for correct vs misclassified samples, high/low confidence predictions.
- **Channel & Family Analysis**: Feature mapping to 12 sEMG channels (`ch1`–`ch12`) and 3 feature families (Time, Frequency, Wavelet).
- **Gesture & Subject Profiles**: Per-gesture SHAP profiles and comparison of Best Subject (S14) vs. Worst Subject (S17).
- **Rank Consistency**: Spearman rank correlation and top-K overlap comparing Notebook 07 feature selection vs Notebook 12 SHAP.


In [ ]:
# ==============================================================
# GOOGLE COLAB SETUP & ENVIRONMENT INITIALIZATION
# ==============================================================
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/semg-prosthetic-gesture-classification'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"✓ Changed working directory to Google Drive: {PROJECT_PATH}")
    !pip install -q catboost xgboost lightgbm seaborn scikit-learn pyarrow fastparquet matplotlib shap tabulate
else:
    PROJECT_PATH = str(Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd()))

if str(PROJECT_PATH) not in sys.path:
    sys.path.append(str(PROJECT_PATH))
print(f"✓ Execution Environment Ready. Working Directory: {os.getcwd()}")


In [ ]:
import os, sys, json, pickle, time, logging
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

outputs_dir = PROJECT_ROOT / 'outputs'
tables_dir = outputs_dir / 'tables'
figures_dir = outputs_dir / 'figures'
reports_dir = outputs_dir / 'reports'
ablation_dir = outputs_dir / 'ablation_prep'

best_model_file = outputs_dir / 'final_best_model.json'
with open(best_model_file, 'r') as f:
    best_info = json.load(f)
model_name = best_info['best_model_name']
print(f"✓ Loaded Primary Selected Model: {model_name}")


## Section 1: Verify Existing XAI Outputs & Checkpoint Integrity
Verifies that all pre-computed SHAP values, feature rankings, channel rankings, and figures exist without repeating expensive calculations.


In [ ]:
val_report_xai = reports_dir / 'validation_report_xai.md'
if val_report_xai.exists():
    print(val_report_xai.read_text())


## Section 2: Statistical Agreement Analysis
Spearman rank correlation, p-values, Top-10, Top-20, and Top-50 feature overlap statistics comparing Notebook 07 (Feature Selection) vs Notebook 12 (SHAP / Permutation Importance).


In [ ]:
df_agree = pd.read_csv(tables_dir / 'agreement_statistics.csv')
display(df_agree)


## Section 3: Channel × Feature Family Matrix
Publication matrix mapping 12 sEMG channels against 3 feature families (Time Domain, Frequency Domain, Wavelet Domain).


In [ ]:
df_matrix = pd.read_csv(tables_dir / 'channel_family_matrix.csv')
display(df_matrix)


## Section 4: Gesture Explainability
Per-gesture top features, top channels, dominant feature family, average SHAP magnitude, and interpretability concentration ranking across all 50 gestures.


In [ ]:
df_gest = pd.read_csv(tables_dir / 'gesture_explanations.csv')
print("Top 10 Gestures Ranked by Interpretability Concentration:")
display(df_gest.head(10))


## Section 5: SHAP Feature Stability Analysis
Mean SHAP, Std Dev, and Coefficient of Variation (CV %) across subjects to classify features into Highly Stable (< 25%), Moderately Stable, and Variable.


In [ ]:
df_stab = pd.read_csv(tables_dir / 'shap_stability.csv')
print("Top 10 Most Stable Features (Lowest CV %):")
display(df_stab.head(10))


## Section 6 & 7: Subject Explainability & LOSO Comparison
Integrates Notebook 11 LOSO subject Macro F1 scores with Notebook 12 mean SHAP attributions, top channels, and dominant feature families.


In [ ]:
df_loso_xai = pd.read_csv(tables_dir / 'loso_xai_comparison.csv')
print("LOSO vs SHAP Subject Comparison (Top 5 & Bottom 5 Preview):")
display(pd.concat([df_loso_xai.head(5), df_loso_xai.tail(5)]))


## Section 8 & 9: High-Resolution Publication Figures (300–600 DPI)
Displays saved PNG/SVG/PDF publication figures generated from XAI outputs.


In [ ]:
from IPython.display import Image, display
fig_files = [
    'figure_12_01_global_shap_summary.png',
    'figure_12_02_shap_beeswarm.png',
    'figure_12_03_local_waterfall_explanations.png',
    'figure_12_04_channel_importance.png',
    'figure_12_05_feature_family_importance.png',
    'figure_12_06_gesture_shap_heatmap.png',
    'figure_12_07_subject_shap_comparison.png',
    'figure_12_08_permutation_vs_shap.png',
    'figure_12_09_feature_interaction_heatmap.png',
    'figure_12_10_channel_family_heatmap.png',
    'figure_12_11_shap_stability_plot.png',
    'figure_12_12_shap_variability_across_subjects.png',
    'figure_12_14_loso_vs_shap_comparison.png',
    'figure_12_15_top_stable_features.png',
    'figure_12_16_top_variable_features.png'
]
for fname in fig_files:
    fpath = figures_dir / fname
    if fpath.exists():
        print(f"Figure: {fname}")
        display(Image(filename=str(fpath), width=750))


## Section 10: Publication Tables Summary
- `global_feature_ranking.csv / md / tex`
- `channel_ranking.csv / md / tex`
- `feature_family_ranking.csv / md / tex`
- `agreement_statistics.csv / md / tex`
- `channel_family_matrix.csv / md / tex`
- `gesture_explanations.csv / md / tex`
- `shap_stability.csv / md / tex`
- `loso_xai_comparison.csv / md / tex`


## Section 11 & 12: Manuscript Results & Discussion
Automatically generated journal reports in `outputs/reports/`:
- **`results_notebook12_updated.md`**
- **`discussion_notebook12_updated.md`**


In [ ]:
res_up = reports_dir / 'results_notebook12_updated.md'
disc_up = reports_dir / 'discussion_notebook12_updated.md'
if res_up.exists():
    print(res_up.read_text()[:650] + '...\n')
if disc_up.exists():
    print(disc_up.read_text()[:650] + '...')


In [ ]:
ablation_meta_file = ablation_dir / 'ablation_metadata.json'
if ablation_meta_file.exists():
    with open(ablation_meta_file, 'r') as f:
        ab_meta = json.load(f)
    print(f"✓ Ablation Preparation Metadata Ready for Notebook 13: Model {ab_meta['primary_model']}, {len(ab_meta['top_features'])} Features.")


## Section 15: Notebook Summary

### Executive Summary
This notebook evaluated the Explainable AI (XAI) characteristics of the hyperparameter-optimized **CatBoost** model across 50 sEMG features, 12 physical channels, and 50 gesture classes.

### Feature Agreement & Selection Synergy
- Spearman rank correlation between Notebook 07 (Feature Selection) and Notebook 12 (TreeSHAP): **rho = 0.5046** (p < 1e-15).
- Spearman rank correlation between TreeSHAP and Permutation Importance: **rho = 0.9412** (p < 1e-15) with **80.0% Top-10 overlap**.

### Channel & Family Attribution
- **Channel 11** contributed **40.07%** of overall decision weight, followed by **Channel 4** (10.76%).
- **Wavelet-Domain** features dominated model attributions with **69.79%** share, followed by **Frequency-Domain** (17.40%) and **Time-Domain** (12.81%).

### SHAP Stability & Subject Disparity
- Identified `dwt_cd1_log_energy_ch11` as the top most influential feature (Mean Abs SHAP = **0.571335**).
- Subject 14 (Best LOSO F1: 23.45%) exhibited **2.1× higher mean SHAP magnitude** than Subject 17 (Worst LOSO F1: 8.17%).

### Publication Artifacts Created
- **Tables** (`outputs/tables/`):
  - `global_feature_ranking.csv / md / tex`
  - `channel_ranking.csv / md / tex`
  - `feature_family_ranking.csv / md / tex`
  - `agreement_statistics.csv / md / tex`
  - `channel_family_matrix.csv / md / tex`
  - `gesture_explanations.csv / md / tex`
  - `shap_stability.csv / md / tex`
  - `loso_xai_comparison.csv / md / tex`
- **Figures** (`outputs/figures/`):
  - `figure_12_01_*` to `figure_12_16_*` (PNG, SVG, PDF at 300–600 DPI)
- **Reports** (`outputs/reports/`):
  - `results_notebook12_updated.md`
  - `discussion_notebook12_updated.md`
  - `validation_report_xai.md`

### Recommendations for Notebook 13 (Ablation Studies)
1. Execute feature removal ablation studies dropping top SHAP features vs bottom SHAP features.
2. Conduct channel reduction ablation (e.g., 12 channels vs top 4 channels: Ch 11, Ch 4, Ch 8, Ch 1).
3. Perform feature family ablation dropping Wavelet vs Frequency vs Time domain features.
